## 1) Import libraries and variables

In [13]:

import os
# Use real data for this script. 
os.environ['USE_SYNTHETIC_INPUTS'] = '0'

import importlib 
import pandas as pd
import numpy as np
import config


from clustering_helpers import get_best_config, build_membership_features, variance_of_means
from config import (
    INPUT_DATA_PATH,
    VALIDATION_RESULTS_PATH,
    VARIANCE_OF_MEANS_PATH,
    ENCODED_MEMBERSHIP_PATH,
    labels_path
)

print(f"Config loaded. USE_SYNTHETIC_INPUTS: {config.USE_SYNTHETIC_INPUTS}")

Config loaded. USE_SYNTHETIC_INPUTS: False


## 2) Load best config & wp3 data

In [14]:
best_config = get_best_config(VALIDATION_RESULTS_PATH)
print(f"Best configuration: {best_config}")

labels_df = pd.read_csv(labels_path(best_config), compression="gzip")
wp3_df = pd.read_csv(INPUT_DATA_PATH)
df = wp3_df.merge(labels_df, on="patient_id", how="inner")
print(f"Merged {len(df)} patients with cluster labels")
display(df.head())


Best configuration: raw_kmedoids_gower
Merged 50 patients with cluster labels


,patient_id,sex,patient_index_date,practice_deregistration_date,death_date,household_size,prostate_cancer,pregnancy,hrtcocp,hf_exclude_date,...,homeless,housebound,birth_date,ethnicity_cat,cat_diabetes,gestationaldm_date,t2dm_date,t1dm_date,otherdm_date,cluster
0,32,female,2019-02-01,NaN,NaN,3553.0,NaN,NaN,NaN,NaN,...,False,False,1937-10-01,White,DM unlikely,NaN,NaN,NaN,NaN,1
1,71,male,2019-02-01,NaN,2023-12-09,NaN,NaN,NaN,NaN,NaN,...,False,False,1933-12-01,Unknown,DM unlikely,NaN,NaN,NaN,NaN,1
2,258,male,2019-02-01,NaN,2021-01-13,-11732.0,NaN,NaN,NaN,NaN,...,False,False,1922-01-01,Mixed,DM unlikely,NaN,NaN,NaN,NaN,1
3,665,male,2019-02-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,1954-02-01,Unknown,DM unlikely,NaN,NaN,NaN,NaN,2
4,700,female,2019-02-01,NaN,NaN,15368.0,NaN,NaN,NaN,NaN,...,False,False,1928-07-01,Unknown,DM unlikely,NaN,NaN,NaN,NaN,1


## 3) Build membership features

In [15]:

membership = build_membership_features(df)
print(f"Built {len(membership.columns)-1} membership features")
display(membership.head())
# Define output directory from VARIANCE_OF_MEANS_PATH
output_dir = os.path.dirname(VARIANCE_OF_MEANS_PATH)
os.makedirs(output_dir, exist_ok=True)

membership_path = os.path.join(output_dir, "membership_features.csv")
membership.to_csv(membership_path, index=False)
print(f"Saved membership features to {membership_path}")



Built 34 membership features


,patient_id,diagnosis_community,diagnosis_emergency,age_band,cat_household_size,sex,ethnicity_cat,imd_quintile,region,rural_urban,...,has_diabetes,mltc_count,has_mltc,carehome_at_index,housebound,smi,homeless,substance_abuse,migrant,non_english_speaking
0,32,0,1,80-84,>=3,female,White,unknown,NaN,NaN,...,0,2,1,0,0,0,0,0,1,0
1,71,0,1,85-89,unknown,male,Unknown,unknown,London,NaN,...,0,1,0,0,0,0,0,0,0,0
2,258,0,1,95-99,unknown,male,Mixed,unknown,South West,NaN,...,0,1,0,0,0,0,0,0,0,0
3,665,0,1,60-64,unknown,male,Unknown,unknown,North West,NaN,...,0,3,1,0,0,0,0,0,0,0
4,700,0,1,90-94,>=3,female,Unknown,unknown,North East,NaN,...,0,0,0,0,0,0,0,0,0,0


Saved membership features to /Users/cu20932/Documents/Work/reducehf/output/clustering/real/membership_features.csv


## 4) One-hot encode features


In [16]:
# One-hot encode categorical variables
X = pd.get_dummies(membership.drop(columns="patient_id"), dummy_na=True, drop_first=False)
display(X.head)
# Identify continuous vs dummy columns
continuous_cols = [col for col in X.columns if col in ['mltc_count']]
dummy_cols = [col for col in X.columns if col not in continuous_cols]

print(f"After one-hot encoding: {len(X.columns)} features "
        f"({len(continuous_cols)} continuous, {len(dummy_cols)} dummy)")

#set type to int 
for col in dummy_cols:
    if col in X.columns:
        X[col] = X[col].astype(int)

# Remove continuous variables and fill NaNs
X = X.drop(columns=[col for col in continuous_vars if col in X.columns])

X.to_csv(ENCODED_MEMBERSHIP_PATH, index=False)
print(f"Saved one-hot encoded features to {encoded_path}")



<bound method NDFrame.head of     diagnosis_community  diagnosis_emergency  copd_preexisting  copd_new  \
0                     0                    1                 0         0   
1                     0                    1                 0         0   
2                     0                    1                 0         0   
3                     0                    1                 0         0   
4                     0                    1                 0         0   
5                     0                    1                 0         0   
6                     0                    1                 0         0   
7                     0                    1                 0         0   
8                     0                    1                 0         0   
9                     0                    1                 0         0   
10                    0                    1                 0         0   
11                    0                    1              

After one-hot encoding: 85 features (1 continuous, 84 dummy)
Saved one-hot encoded features to /Users/cu20932/Documents/Work/reducehf/output/clustering/real/membership_features_encoded.csv


## 5) Compute variance-of-means + export

In [17]:
# Calculate variance of means
vom = variance_of_means(df["cluster"], X)
vom.sort_values(ascending=False).to_csv(VARIANCE_OF_MEANS_PATH, header=["variance_of_means"])

print(f"\nSaved: {VARIANCE_OF_MEANS_PATH}")
print(f"\nTop 10 most discriminative features:")
print(vom.sort_values(ascending=False).head(10))


Saved: /Users/cu20932/Documents/Work/reducehf/output/clustering/real/variance_of_means.csv

Top 10 most discriminative features:
cat_diabetes_GDM            0.075996
cat_diabetes_DM unlikely    0.058761
has_diabetes                0.058761
sex_male                    0.052901
sex_female                  0.052901
hypertension_new            0.034525
cat_diabetes_T2DM           0.032410
region_South East           0.032410
af_preexisting              0.032410
age_band_95-99              0.032410
dtype: float64


In [18]:
display(labels_df.head())

,patient_id,cluster
0,18,1
1,19,1
2,25,2
3,30,1
4,32,1


In [19]:
display(wp3_df.head())

,patient_id,sex,patient_index_date,practice_deregistration_date,death_date,household_size,prostate_cancer,pregnancy,hrtcocp,hf_exclude_date,...,substance_abuse,homeless,housebound,birth_date,ethnicity_cat,cat_diabetes,gestationaldm_date,t2dm_date,t1dm_date,otherdm_date
0,32,female,2019-02-01,NaN,NaN,3553.0,NaN,NaN,NaN,NaN,...,False,False,False,1937-10-01,White,DM unlikely,NaN,NaN,NaN,NaN
1,58,male,2019-02-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,False,False,1962-12-01,Unknown,DM unlikely,NaN,NaN,NaN,NaN
2,66,female,2019-02-01,NaN,2021-09-01,NaN,NaN,NaN,NaN,NaN,...,False,False,False,1962-04-01,Unknown,T1DM,NaN,NaN,2021-04-27,NaN
3,71,male,2019-02-01,NaN,2023-12-09,NaN,NaN,NaN,NaN,NaN,...,False,False,False,1933-12-01,Unknown,DM unlikely,NaN,NaN,NaN,NaN
4,127,female,2019-02-01,NaN,NaN,19004.0,NaN,NaN,NaN,NaN,...,False,False,False,1942-07-01,White,DM unlikely,NaN,NaN,NaN,NaN
